In [ ]:
# Setup the Jupyter version of Dash
from dash import Dash, html, dcc, dash_table

# Dashboard component imports
import dash_leaflet as dl
from dash.dependencies import Input, Output
import plotly.express as px
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# CRUD Python module (file name and class name) - Atlas via MONGO_URI
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################


# Connect to database via CRUD Module
# animal_shelter.py handles database (AAC) and collection (animals)
# and reads MongoDB Atlas connection string from MONGO_URI
shelter = AnimalShelter()

# Populate initial DataTable by reading all documents
# Use projection enhancement to exclude MongoDB "_id" (ObjectId can cause DataTable issues)
df = pd.DataFrame.from_records(shelter.read({}, projection={"_id": 0}))

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

# Added in Grazioso Salvare’s logo
image_filename = 'Grazioso_Salvare_Logo.png' # matches name and path of image
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

# Place the HTML image tag in the line below into the app.layout code according to your design
# Also remember to include a unique identifier such as your name or date

app.layout = html.Div([
    #html.Div(id='hidden-div', style={'display':'none'}),
    # Add logo with a link
    html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image), style={'height' : '10%', 'width' : '10%'}), href='http://www.snhu.edu'),
    html.Center(html.B(html.H1("CS-340 Dashboard - Pi'ilani Apo"))),
    html.Hr(),
    # Filter Controls
    html.Div([
        # Add interactive filtering options
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset'
        )
    ]),
    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_current=0,
        page_size=10,
        page_action="native",
        row_selectable="single",
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        style_table={'overflowX' : 'auto'}
    ),
    html.Br(),
    html.Hr(),

    # Set up dashboard so chart and geolocation are side-by-side
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Filter interactive data table with MongoDB queries
    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamut", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        # Reset/Default: Show all documents
        query = {}
    
    data = pd.DataFrame.from_records(shelter.read(query, projection={"_id":0}))
    return data.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Add code for chart of your choice (e.g. pie chart)
    if viewData is None or len(viewData) == 0:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)

    # If no breed in column exists in the view, do not render graph
    if "breed" not in dff.columns:
        return []

    fig = px.pie(dff, names='breed', title='Preferred Animals by breed')
    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # Geolocation chart
    # Check if viewData is None or empty
    if viewData is None or len(viewData) == 0:
        return []
    
    # Convert viewData to DataFrame
    dff = pd.DataFrame.from_dict(viewData)
    
    # Check if index is None or empty + default to first row if none selected
    row = 0
    if index is not None and len(index) > 0:
        row = index[0]

    # Use column names for dataset; changed from iloc column positions
    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return []

    lat = dff.loc[row, "location_lat"]
    lon = dff.loc[row, "location_long"]

    # In case of missing coordinates
    if pd.isna(lat) or pd.isna(lon):
        return []

    tooltip_text = dff.loc[row, "breed"] if "breed" in dff.columns else "Animal"
    animal_name = dff.loc[row, "name"] if "name" in dff.columns else ""

    # Austin TX
    return [
        dl.Map(
            style={'width' : '1000px', 'height' : '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(tooltip_text),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(animal_name)
                        ])
                    ]
                )
            ]
        )
    ]

app.run(debug=True, host="0.0.0.0", port=8050)

In [2]:
from animal_shelter import AnimalShelter
shelter = AnimalShelter()
docs = shelter.read({}, projection={"_id": 0})
print("docs returned:", len(docs))
print("first doc keys:", list(docs[0].keys()) if docs else "no docs")

docs returned: 10000
first doc keys: ['1', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks']
